# 🚀 Real-Debrid ➔ Google Drive (Sem Estourar os 107 GB do Colab)

Este notebook foi feito especialmente para lidar com **cursos pesados (80 GB+)** em ambientes com pouco espaço:
- 🔄 **Processamento em Partes**: Se o torrent tiver vários arquivos/módulos, baixa 1 por vez, descompacta no Drive e **deleta imediatamente** antes de baixar o próximo (nunca lota o Colab).
- ⚡ **Streaming Unzip (0 MB de disco)**: Se for um único arquivo `.zip` de 80 GB, descompacta **direto da nuvem para o Google Drive** sem baixar o arquivo `.zip` no Colab!
- 📊 **Velocidade em tempo real**: Mostra taxa de download (MB/s), progresso e tempo restante.

### 1️⃣ Passo 1: Montar o Google Drive e Instalar Ferramentas

In [ ]:
# Conecta seu Google Drive (autorize na janela pop-up)
from google.colab import drive
drive.mount('/content/drive')

# Instala o aria2 e 7-Zip
!apt-get update -qq && apt-get install -y aria2 p7zip-full -qq
print("\n✅ Google Drive conectado e ferramentas prontas!")

### 2️⃣ Passo 2: Executar Transferência Inteligente
Escolha o modo de extração de acordo com o seu arquivo.

In [ ]:
import os
import sys
import io
import time
import re
import shutil
import subprocess
import zipfile
import requests

# ================= CONFIGURAÇÕES =================
RD_API_TOKEN = "COLE_SEU_TOKEN_AQUI"            # https://real-debrid.com/apitoken
LINK_OU_MAGNET = "COLE_SEU_LINK_OU_MAGNET_AQUI" # Magnet ou Link do Real-Debrid
PASTA_NO_DRIVE = "Cursos"                      # Nome da pasta no seu Google Drive

# Para um ÚNICO arquivo .zip gigante (ex: 80GB), use True para descompactar direto via streaming sem salvar o zip!
MODO_STREAMING_ZIP = True  
# =================================================

BASE_URL = "https://api.real-debrid.com/rest/1.0"
HEADERS = {"Authorization": f"Bearer {RD_API_TOKEN.strip()}"}
ARCHIVE_EXTS = (".zip", ".rar", ".7z", ".tar", ".tar.gz", ".tgz", ".tar.bz2")

# 1. Valida conta do Real-Debrid
user_resp = requests.get(f"{BASE_URL}/user", headers=HEADERS)
if user_resp.status_code != 200:
    raise SystemExit("❌ Token inválido ou expirado no Real-Debrid!")
user_data = user_resp.json()
print(f"👤 Conectado como: {user_data.get('username')} ({user_data.get('type')})")

# 2. Pastas
temp_dir = "/content/temp_download"
drive_destino = os.path.join("/content/drive/MyDrive", PASTA_NO_DRIVE.strip("/"))
os.makedirs(temp_dir, exist_ok=True)
os.makedirs(drive_destino, exist_ok=True)

# 3. Obtenção dos links
download_items = []
link_input = LINK_OU_MAGNET.strip()

if link_input.startswith("magnet:"):
    print("\n🧲 Enviando magnet para o Real-Debrid...")
    torrent_id = requests.post(f"{BASE_URL}/torrents/addMagnet", headers=HEADERS, data={"magnet": link_input}).json()["id"]
    requests.post(f"{BASE_URL}/torrents/selectFiles/{torrent_id}", headers=HEADERS, data={"files": "all"})
    
    while True:
        info = requests.get(f"{BASE_URL}/torrents/info/{torrent_id}", headers=HEADERS).json()
        status = info.get("status")
        prog = info.get("progress", 0)
        if status == "downloaded":
            print(f"✅ Pronto no Real-Debrid: {info.get('filename')}")
            break
        elif status in ["error", "virus", "dead"]:
            raise SystemExit(f"❌ Erro no torrent no Real-Debrid: {status}")
        else:
            print(f"⏳ Processando no Real-Debrid: {status} ({prog}%)...", end="\r")
            time.sleep(3)
            
    for l in info.get("links", []):
        unr = requests.post(f"{BASE_URL}/unrestrict/link", headers=HEADERS, data={"link": l}).json()
        if unr.get("download"):
            download_items.append((unr.get("filename"), unr.get("download")))
else:
    unr = requests.post(f"{BASE_URL}/unrestrict/link", headers=HEADERS, data={"link": link_input}).json()
    if unr.get("download"):
        download_items.append((unr.get("filename"), unr.get("download")))
    else:
        nome = link_input.split("/")[-1].split("?")[0] or "arquivo"
        download_items.append((nome, link_input))

print(f"\n📥 Total de itens a processar: {len(download_items)}")

# -----------------------------------------------------
# Leitor HTTP Range para Streaming Unzip (0 MB de disco)
# -----------------------------------------------------
class HTTPSeekableStream(io.RawIOBase):
    def __init__(self, url):
        self.url = url
        self.session = requests.Session()
        self.pos = 0
        r = self.session.head(url, allow_redirects=True)
        self.size = int(r.headers.get("Content-Length", 0))
        self.buf = b""
        self.buf_start = 0

    def seekable(self): return True
    def readable(self): return True
    def seek(self, offset, whence=io.SEEK_SET):
        if whence == io.SEEK_SET: self.pos = offset
        elif whence == io.SEEK_CUR: self.pos += offset
        elif whence == io.SEEK_END: self.pos = self.size + offset
        return self.pos
    def tell(self): return self.pos

    def readinto(self, b):
        if self.pos >= self.size: return 0
        needed = len(b)
        if self.buf_start <= self.pos < self.buf_start + len(self.buf):
            offset = self.pos - self.buf_start
            avail = len(self.buf) - offset
            n = min(needed, avail)
            b[:n] = self.buf[offset:offset+n]
            self.pos += n
            return n
        fetch_len = max(needed, 2 * 1024 * 1024)  # buffer de 2MB por requisição
        end = min(self.pos + fetch_len - 1, self.size - 1)
        r = self.session.get(self.url, headers={"Range": f"bytes={self.pos}-{end}"})
        if r.status_code == 206:
            self.buf = r.content
        elif r.status_code == 200:
            self.buf = r.content[self.pos:end+1]
        else:
            return 0
        self.buf_start = self.pos
        n = min(needed, len(self.buf))
        b[:n] = self.buf[:n]
        self.pos += n
        return n

def extrair_zip_streaming(url, destino_pasta):
    print("\n🌊 Modo Streaming ativado: Extraindo arquivos direto da nuvem para o Google Drive...")
    print("💡 O arquivo .zip NÃO será salvo no disco! Uso de disco do Colab = 0 MB.")
    stream = io.BufferedReader(HTTPSeekableStream(url), buffer_size=1024*1024)
    with zipfile.ZipFile(stream) as zf:
        infolist = [m for m in zf.infolist() if not m.is_dir()]
        total_files = len(infolist)
        print(f"📑 {total_files} arquivos encontrados dentro do ZIP remoto.\n")
        
        for idx, member in enumerate(infolist, 1):
            out_path = os.path.join(destino_pasta, member.filename)
            os.makedirs(os.path.dirname(out_path), exist_ok=True)
            
            file_mb = member.file_size / (1024 * 1024)
            print(f"[{idx}/{total_files}] 📥 Gravando no Drive: {member.filename} ({file_mb:.1f} MB)")
            
            start_t = time.time()
            written = 0
            with zf.open(member) as src, open(out_path, "wb") as dst:
                while True:
                    chunk = src.read(512 * 1024)
                    if not chunk: break
                    dst.write(chunk)
                    written += len(chunk)
                    elapsed = max(time.time() - start_t, 0.001)
                    speed = (written / (1024 * 1024)) / elapsed
                    perc = (written / member.file_size * 100) if member.file_size else 100
                    sys.stdout.write(f"\r   ⚡ Progresso: {written/(1024*1024):.1f}/{file_mb:.1f} MB ({perc:.0f}%) | Velocidade: {speed:.1f} MB/s   ")
                    sys.stdout.flush()
            sys.stdout.write("\r" + " " * 80 + "\r")
            print(f"   ✅ Salvo no Google Drive: {member.filename}")

# -----------------------------------------------------
# Download Convencional de alta velocidade com aria2c
# -----------------------------------------------------
def baixar_com_velocidade(url, destino_pasta, nome_arquivo):
    cmd = [
        "aria2c", "-x", "16", "-s", "16", "-k", "1M",
        "--summary-interval=1", "--show-console-readout=true", "--console-log-level=warn",
        "--file-allocation=none", "--dir", destino_pasta, "-o", nome_arquivo, url
    ]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    regex = re.compile(r"\[#\w+\s+([\d\.\w]+)/([\d\.\w]+)\(([\d%]+)\)\s+CN:\d+\s+DL:([\d\.\w]+)(?:\s+ETA:([\d\.\w]+))?\]")
    
    for line in proc.stdout:
        m = regex.search(line)
        if m:
            baixado, total, perc, vel, eta = m.groups()
            eta_str = f" | Restante: {eta}" if eta else ""
            sys.stdout.write(f"\r🚀 Progresso: {baixado}/{total} ({perc}) | Velocidade: {vel}/s{eta_str}   ")
            sys.stdout.flush()
    proc.wait()
    sys.stdout.write("\r" + " " * 90 + "\r")
    if proc.returncode != 0:
        raise RuntimeError(f"Erro ao baixar {nome_arquivo}")

# 4. Execução Principal: Processamento Inteligente
for idx, (nome, url) in enumerate(download_items, 1):
    print(f"\n{'='*60}")
    print(f"[{idx}/{len(download_items)}] 📌 Processando: {nome}")
    print(f"{'='*60}")
    
    nome_lower = nome.lower()
    
    # CASO 1: É um arquivo .ZIP e MODO_STREAMING está ativo (economiza 100% do disco do Colab)
    if nome_lower.endswith(".zip") and MODO_STREAMING_ZIP:
        nome_pasta = re.sub(r"\.zip$", "", nome, flags=re.I)
        pasta_curso = os.path.join(drive_destino, nome_pasta)
        try:
            extrair_zip_streaming(url, pasta_curso)
            continue
        except Exception as e:
            print(f"⚠️ Streaming direto não suportado por este ZIP ({e}). Alternando para download em partes...")
    
    # CASO 2: É compactado (.rar, .7z, .zip sem streaming)
    if nome_lower.endswith(ARCHIVE_EXTS):
        caminho_temp = os.path.join(temp_dir, nome)
        if os.path.exists(caminho_temp): os.remove(caminho_temp)
        
        print(f"⚡ Baixando parte compactada para extração...")
        baixar_com_velocidade(url, temp_dir, nome)
        
        nome_pasta = re.sub(r"\.(zip|rar|7z|tar\.gz|tgz|tar)$", "", nome, flags=re.I)
        pasta_curso = os.path.join(drive_destino, nome_pasta)
        os.makedirs(pasta_curso, exist_ok=True)
        
        print(f"📦 Descompactando direto no Google Drive em: {pasta_curso} ...")
        cmd_7z = ["7z", "x", caminho_temp, f"-o{pasta_curso}", "-y", "-bsp1"]
        proc_7z = subprocess.Popen(cmd_7z, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in proc_7z.stdout:
            if "%" in line:
                sys.stdout.write(f"\r📦 Extraindo: {line.strip()[:60]}   ")
                sys.stdout.flush()
        proc_7z.wait()
        sys.stdout.write("\r" + " " * 70 + "\r")
        print("✅ Descompactado no Google Drive!")
        
        # DELETA IMEDIATAMENTE O ARQUIVO DO COLAB PARA LIBERAR OS 107 GB!
        if os.path.exists(caminho_temp):
            os.remove(caminho_temp)
            print(f"🗑️ Arquivo temporário {nome} removido do Colab (espaço em disco liberado!)")
            
    else:
        # CASO 3: Arquivo normal (vídeo .mp4, pdf, etc.) - baixa direto pro Drive
        baixar_com_velocidade(url, drive_destino, nome)
        print(f"✅ Salvo no Google Drive: {nome}")

print("\n🎉 Sucesso absoluto! Todos os arquivos foram transferidos e organizados no seu Google Drive sem estourar o limite de disco.")